# JUG quick start

The core JUG Python API in six steps, on J1909-3744 (MeerKAT, 2828 TOAs).
The par file carries a noise model (EFAC, EQUAD, ECORR, power-law DM noise),
so `fit_parameters()` runs a generalised least-squares fit.

## 0. Backend

JUG runs on whatever JAX backend is installed. This example pins the CPU
backend so it is reproducible anywhere; export `JAX_PLATFORMS=cuda` before
starting the kernel to run on a GPU instead.

In [1]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")  # must be set before JAX is imported

'cpu'

## 1. Open a session

In [2]:
from jug.engine.session import TimingSession

session = TimingSession("../tests/data_golden/J1909_parity_noise.par",
                        "../tests/data_golden/J1909_parity.tim")
print(session)

TimingSession(par='J1909_parity_noise.par', tim='J1909_parity.tim', ntoas=2828)


## 2. Compute pre-fit residuals

In [3]:
pre = session.compute_residuals()
print(f"pre-fit RMS = {pre['rms_us']:.3f} us  ({pre['n_toas']} TOAs)")
print("free parameters:", session.free_params)

pre-fit RMS = 0.377 us  (2828 TOAs)
free parameters: ['A1', 'DECJ', 'DM1', 'DM2', 'EPS1', 'EPS2', 'F0', 'F1', 'FD1', 'FD2', 'FD3', 'FD4', 'FD5', 'FD6', 'FD7', 'FD8', 'FD9', 'M2', 'PB', 'PBDOT', 'PMDEC', 'PMRA', 'PX', 'RAJ', 'SINI', 'TASC', 'XDOT']


## 3. Fit the timing model

`fit_parameters()` fits every parameter flagged free in the par file and
auto-detects the noise model, so this is a GLS fit.

In [4]:
fit = session.fit_parameters()
print(f"post-fit RMS = {fit['final_rms']:.3f} us   chi2 = {fit['final_chi2']:.1f}")
print(f"{fit['iterations']} iterations, converged={fit['converged']}")

[FITTER] noise_config was None, auto-detecting from par file
[SETUP] Building DM NOISE basis: 60 columns
post-fit RMS = 0.497 us   chi2 = 12533.0
48 iterations, converged=True


Pre/post-fit parameter comparison with uncertainties:

In [5]:
session.parameter_table(fit)

Parameter                     Pre-fit               Post-fit    Uncertainty  Delta/sigma
----------------------------------------------------------------------------------------
RAJ                    5.016907101599       5.01690710139602   2.943822e-10         0.69
DECJ               -0.658643187230165     -0.658643187124504   7.137439e-10         0.15
PMRA                 -9.5414007566604      -9.52748060366656   1.935216e-02         0.72
PMDEC               -35.6705672645007      -35.7231457577601   6.592587e-02         0.80
F0                   339.315691919041       339.315691919041   7.967200e-13         0.78
F1              -1.61474003828438e-15  -1.61475826658383e-15   1.407852e-20         1.29
DM1              5.81904557819087e-06   4.91880135965184e-05   5.164680e-05         0.84
DM2             -5.45996439805065e-05  -5.42117665934684e-05   2.811456e-05         0.01
PX                  0.959805015591808      0.919713663128298   1.446968e-01         0.28
PB                   

## 4. Estimate the noise model (MAP)

Stochastic-parameter estimation by SVI at fixed timing model. The default
estimates EFAC, EQUAD, ECORR, red noise and DM noise.

In [6]:
est = session.estimate_noise()
for name, value in est.params.items():
    print(f"{name:<16} {value:>10.4f}")

EFAC_KAT_MKBF        1.0530
EQUAD_KAT_MKBF       0.0293
ECORR_KAT_MKBF       0.0920
TNREDAMP           -13.9922
TNREDGAM             1.8905
TNREDC              30.0000
TNDMAMP            -13.4716
TNDMGAM              1.7299
TNDMC               30.0000


Pass any estimator option through to choose what is estimated — e.g. white
noise and DM noise only, with a shorter SVI run:

In [7]:
white_dm = session.estimate_noise(include_red_noise=False, max_num_batches=10)
print(list(white_dm.params))

['EFAC_KAT_MKBF', 'EQUAD_KAT_MKBF', 'ECORR_KAT_MKBF', 'TNDMAMP', 'TNDMGAM', 'TNDMC']


## 5. Write the post-fit ephemeris

In [8]:
session.save_par("J1909-3744_postfit.par", fit_result=fit)
print(open("J1909-3744_postfit.par").read()[:400])

# Created by JUG on 2026-08-03 18:55:36
#
EPHEM                   DE440
CLK                     TT(BIPM2024)
UNITS                   TDB
TIMEEPH                 FB90
T2CMETHOD               IAU2000B
DILATEFREQ              N
NTOA                    2828.0
START                   58526.21388912177
FINISH                  60837.85782723828
PLANET_SHAPIRO          Y
CORRECT_TROPOSPHERE     Y
NE_SW   
